# Day 056 | Shear Wave Velocity from SPT (ML)

## 100 Days of ML for Geotechnical Engineering
**Phase:** Classical Machine Learning

**Author:** Ripon Chandra Malo  
**Date:** YYYY-MM-DD  

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| Vs from SPT | Predict shear wave velocity from field tests | Seismic site classification without MASW |
| Empirical Correlations | Traditional Imai & Tonouchi (1982) equations | Vs = 97 * N^0.314 |
| ML Regression Models | Data-driven Vs prediction | RF, GB, SVR using multiple features |
| Multi-feature Advantage | ML uses depth, soil type, fines — not just N | More information = better prediction |
| Model Comparison | Systematic evaluation of multiple approaches | Which method gives lowest RMSE? |
| Residual Analysis | Check where predictions fail | Does ML improve on empirical at extremes? |

---

## Our Geotechnical Problem

**Shear wave velocity (Vs)** is critical for seismic site classification, liquefaction assessment, and dynamic analysis.  
Direct measurement (MASW, crosshole) is expensive. SPT is done on nearly every project.  
Can ML predict Vs from SPT data better than traditional empirical correlations?

---

## Part 1: Load and Explore SPT-Vs Dataset

In [ ]:
# ─────────────────────────────────────────────────────
# LOAD DATA AND INITIAL EXPLORATION
# ─────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('spt_vs_data.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nSoil types: {df['soil_type'].value_counts().to_dict()}")
print(f"\nTarget (Vs_measured) range: {df['Vs_measured'].min():.0f} - {df['Vs_measured'].max():.0f} m/s")
print(f"\nDescriptive statistics:")
print(df.describe().round(2))

In [ ]:
# ─────────────────────────────────────────────────────
# EXPLORATORY PLOTS
# ─────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# N60 vs Vs
for soil in df['soil_type'].unique():
    mask = df['soil_type'] == soil
    axes[0, 0].scatter(df.loc[mask, 'N60'], df.loc[mask, 'Vs_measured'],
                        alpha=0.6, s=30, label=soil)
axes[0, 0].set_xlabel('N60', fontsize=12)
axes[0, 0].set_ylabel('Vs (m/s)', fontsize=12)
axes[0, 0].set_title('N60 vs Shear Wave Velocity', fontsize=13, fontweight='bold')
axes[0, 0].legend(fontsize=9)

# Depth vs Vs
axes[0, 1].scatter(df['depth'], df['Vs_measured'], alpha=0.5, c='#3498db', s=30)
axes[0, 1].set_xlabel('Depth (m)', fontsize=12)
axes[0, 1].set_ylabel('Vs (m/s)', fontsize=12)
axes[0, 1].set_title('Depth vs Shear Wave Velocity', fontsize=13, fontweight='bold')

# Vs distribution
axes[1, 0].hist(df['Vs_measured'], bins=25, color='#2ecc71', edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Vs (m/s)', fontsize=12)
axes[1, 0].set_ylabel('Frequency', fontsize=12)
axes[1, 0].set_title('Vs Distribution', fontsize=13, fontweight='bold')

# Boxplot by soil type
soil_types = df['soil_type'].unique()
vs_by_soil = [df.loc[df['soil_type'] == s, 'Vs_measured'].values for s in soil_types]
axes[1, 1].boxplot(vs_by_soil, labels=soil_types)
axes[1, 1].set_ylabel('Vs (m/s)', fontsize=12)
axes[1, 1].set_title('Vs by Soil Type', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

---

## Part 2: Empirical Correlations (Imai & Tonouchi)

In [ ]:
# ─────────────────────────────────────────────────────
# EMPIRICAL CORRELATIONS — Imai & Tonouchi (1982)
# ─────────────────────────────────────────────────────

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Imai & Tonouchi (1982): Vs = 97 * N^0.314
df['Vs_imai'] = 97 * df['N_value'] ** 0.314

# Alternative: Ohta & Goto (1978): Vs = 85.35 * N^0.348
df['Vs_ohta'] = 85.35 * df['N_value'] ** 0.348

# Evaluate empirical correlations
for name, col in [('Imai & Tonouchi', 'Vs_imai'), ('Ohta & Goto', 'Vs_ohta')]:
    r2 = r2_score(df['Vs_measured'], df[col])
    rmse = np.sqrt(mean_squared_error(df['Vs_measured'], df[col]))
    mae = mean_absolute_error(df['Vs_measured'], df[col])
    print(f"{name:20s}: R2={r2:.4f}, RMSE={rmse:.1f} m/s, MAE={mae:.1f} m/s")

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(df['Vs_measured'], df['Vs_imai'], alpha=0.5, label='Imai & Tonouchi', color='#3498db')
ax.scatter(df['Vs_measured'], df['Vs_ohta'], alpha=0.5, label='Ohta & Goto', color='#e67e22')
lims = [df['Vs_measured'].min()-20, df['Vs_measured'].max()+20]
ax.plot(lims, lims, 'k--', linewidth=2, label='Perfect')
ax.set_xlabel('Measured Vs (m/s)', fontsize=12)
ax.set_ylabel('Predicted Vs (m/s)', fontsize=12)
ax.set_title('Empirical Correlations: Predicted vs Measured', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(lims)
ax.set_ylim(lims)
plt.tight_layout()
plt.show()

print("\nEmpirical correlations use ONLY N-value.")
print("ML can leverage depth, soil type, fines content, and more.")

---

## Part 3: Prepare Data for ML Models

In [ ]:
# ─────────────────────────────────────────────────────
# FEATURE ENGINEERING AND TRAIN/TEST SPLIT
# ─────────────────────────────────────────────────────

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Encode soil type
le = LabelEncoder()
df['soil_type_encoded'] = le.fit_transform(df['soil_type'])

features = ['depth', 'N_value', 'N60', 'soil_type_encoded',
            'fines_content', 'unit_weight', 'effective_stress']
target = 'Vs_measured'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")
print(f"Features used:    {features}")

---

## Part 4: Train Multiple ML Models

In [ ]:
# ─────────────────────────────────────────────────────
# TRAIN AND COMPARE MULTIPLE ML MODELS
# ─────────────────────────────────────────────────────

from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                               AdaBoostRegressor)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_score

models = {
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.1),
    'KNN': KNeighborsRegressor(n_neighbors=7),
    'SVR': SVR(kernel='rbf', C=100, epsilon=5),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42),
    'AdaBoost': AdaBoostRegressor(n_estimators=100, random_state=42)
}

results = {}
predictions = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    predictions[name] = y_pred
    cv = cross_val_score(model, X_train_s, y_train, cv=5, scoring='r2')
    results[name] = {
        'CV R2': cv.mean(),
        'Test R2': r2_score(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'MAE': mean_absolute_error(y_test, y_pred)
    }

results_df = pd.DataFrame(results).T.round(4).sort_values('Test R2', ascending=False)
print("ML Model Comparison:")
print("=" * 65)
print(results_df)

---

## Part 5: ML vs Empirical — Head-to-Head Comparison

In [ ]:
# ─────────────────────────────────────────────────────
# ML vs EMPIRICAL COMPARISON
# ─────────────────────────────────────────────────────

best_model_name = results_df.index[0]
best_model = models[best_model_name]
y_pred_best = predictions[best_model_name]

# Get empirical predictions for test set
test_idx = X_test.index
vs_imai_test = df.loc[test_idx, 'Vs_imai'].values

# Summary comparison
comparison = pd.DataFrame({
    'Method': ['Imai & Tonouchi', f'ML ({best_model_name})'],
    'R2': [r2_score(y_test, vs_imai_test), r2_score(y_test, y_pred_best)],
    'RMSE (m/s)': [np.sqrt(mean_squared_error(y_test, vs_imai_test)),
                   np.sqrt(mean_squared_error(y_test, y_pred_best))],
    'MAE (m/s)': [mean_absolute_error(y_test, vs_imai_test),
                  mean_absolute_error(y_test, y_pred_best)]
}).round(4)

print("HEAD-TO-HEAD: Empirical vs ML")
print("=" * 55)
print(comparison.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Empirical
axes[0].scatter(y_test, vs_imai_test, alpha=0.6, color='#3498db',
                edgecolors='black', s=40)
axes[0].plot(lims, lims, 'k--', linewidth=2)
axes[0].set_xlabel('Measured Vs (m/s)', fontsize=12)
axes[0].set_ylabel('Predicted Vs (m/s)', fontsize=12)
r2_emp = r2_score(y_test, vs_imai_test)
axes[0].set_title(f'Imai & Tonouchi (R2={r2_emp:.3f})', fontsize=13, fontweight='bold')

# Best ML
axes[1].scatter(y_test, y_pred_best, alpha=0.6, color='#e74c3c',
                edgecolors='black', s=40)
axes[1].plot(lims, lims, 'k--', linewidth=2)
axes[1].set_xlabel('Measured Vs (m/s)', fontsize=12)
axes[1].set_ylabel('Predicted Vs (m/s)', fontsize=12)
r2_ml = r2_score(y_test, y_pred_best)
axes[1].set_title(f'{best_model_name} (R2={r2_ml:.3f})', fontsize=13, fontweight='bold')

plt.suptitle('Empirical vs ML: Shear Wave Velocity Prediction',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## Part 6: Residual Analysis

In [ ]:
# ─────────────────────────────────────────────────────
# RESIDUAL ANALYSIS — Where do models fail?
# ─────────────────────────────────────────────────────

residuals_emp = y_test.values - vs_imai_test
residuals_ml = y_test.values - y_pred_best

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Residuals vs Measured (Empirical)
axes[0, 0].scatter(y_test, residuals_emp, alpha=0.6, color='#3498db', s=30)
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Measured Vs (m/s)', fontsize=11)
axes[0, 0].set_ylabel('Residual (m/s)', fontsize=11)
axes[0, 0].set_title('Empirical Residuals', fontsize=12, fontweight='bold')

# Residuals vs Measured (ML)
axes[0, 1].scatter(y_test, residuals_ml, alpha=0.6, color='#e74c3c', s=30)
axes[0, 1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Measured Vs (m/s)', fontsize=11)
axes[0, 1].set_ylabel('Residual (m/s)', fontsize=11)
axes[0, 1].set_title('ML Residuals', fontsize=12, fontweight='bold')

# Residual histograms
axes[1, 0].hist(residuals_emp, bins=20, alpha=0.6, color='#3498db',
                edgecolor='black', label=f'Empirical (std={residuals_emp.std():.1f})')
axes[1, 0].hist(residuals_ml, bins=20, alpha=0.6, color='#e74c3c',
                edgecolor='black', label=f'ML (std={residuals_ml.std():.1f})')
axes[1, 0].set_xlabel('Residual (m/s)', fontsize=11)
axes[1, 0].set_ylabel('Frequency', fontsize=11)
axes[1, 0].set_title('Residual Distribution', fontsize=12, fontweight='bold')
axes[1, 0].legend(fontsize=10)

# Absolute error by soil type
test_soils = df.loc[test_idx, 'soil_type'].values
err_data = pd.DataFrame({
    'soil_type': test_soils,
    'Empirical |Error|': np.abs(residuals_emp),
    'ML |Error|': np.abs(residuals_ml)
})
err_summary = err_data.groupby('soil_type').mean().round(1)
err_summary.plot(kind='bar', ax=axes[1, 1], color=['#3498db', '#e74c3c'], edgecolor='black')
axes[1, 1].set_ylabel('Mean Absolute Error (m/s)', fontsize=11)
axes[1, 1].set_title('Error by Soil Type', fontsize=12, fontweight='bold')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.suptitle('Residual Analysis: Empirical vs ML',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## Part 7: Feature Importance for Vs Prediction

In [ ]:
# ─────────────────────────────────────────────────────
# FEATURE IMPORTANCE — What drives Vs prediction?
# ─────────────────────────────────────────────────────

from sklearn.inspection import permutation_importance

# Use RF for feature importance
rf = models['Random Forest']
perm = permutation_importance(rf, X_test_s, y_test, n_repeats=20, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# MDI importance
sorted_mdi = np.argsort(rf.feature_importances_)
axes[0].barh(np.array(features)[sorted_mdi], rf.feature_importances_[sorted_mdi],
             color='#3498db', edgecolor='black')
axes[0].set_xlabel('MDI Importance', fontsize=12)
axes[0].set_title('Tree-Based Importance', fontsize=13, fontweight='bold')

# Permutation importance
sorted_perm = np.argsort(perm.importances_mean)
axes[1].barh(np.array(features)[sorted_perm], perm.importances_mean[sorted_perm],
             color='#e74c3c', edgecolor='black',
             xerr=perm.importances_std[sorted_perm])
axes[1].set_xlabel('Permutation Importance (R2 drop)', fontsize=12)
axes[1].set_title('Permutation Importance', fontsize=13, fontweight='bold')

plt.suptitle('Feature Importance for Vs Prediction',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Empirical methods use ONLY N-value.")
print("ML leverages all features — that is why it often outperforms empirical.")

---

## Part 8: Seismic Site Classification Comparison

In [ ]:
# ─────────────────────────────────────────────────────
# SEISMIC SITE CLASS — Does Vs error affect classification?
# ─────────────────────────────────────────────────────

def site_class(vs):
    """NEHRP site classification based on Vs30."""
    if vs > 760: return 'A'
    elif vs > 360: return 'B'
    elif vs > 180: return 'C'
    elif vs > 100: return 'D'
    else: return 'E'

actual_class = [site_class(v) for v in y_test]
emp_class = [site_class(v) for v in vs_imai_test]
ml_class = [site_class(v) for v in y_pred_best]

from sklearn.metrics import accuracy_score

emp_accuracy = accuracy_score(actual_class, emp_class)
ml_accuracy = accuracy_score(actual_class, ml_class)

print("Site Classification Accuracy (based on Vs predictions):")
print(f"  Empirical (Imai & Tonouchi): {emp_accuracy*100:.1f}%")
print(f"  ML ({best_model_name}):       {ml_accuracy*100:.1f}%")
print()

# Confusion visualization
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
classes = sorted(set(actual_class))

cm_emp = confusion_matrix(actual_class, emp_class, labels=classes)
ConfusionMatrixDisplay(cm_emp, display_labels=classes).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'Empirical (Acc={emp_accuracy:.1%})', fontsize=13, fontweight='bold')

cm_ml = confusion_matrix(actual_class, ml_class, labels=classes)
ConfusionMatrixDisplay(cm_ml, display_labels=classes).plot(ax=axes[1], cmap='Reds', colorbar=False)
axes[1].set_title(f'ML (Acc={ml_accuracy:.1%})', fontsize=13, fontweight='bold')

plt.suptitle('Seismic Site Classification from Predicted Vs',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## Practice Exercises

### Exercise 1: Soil-Specific Models
Train separate ML models for each soil type. Does specialization improve accuracy?

### Exercise 2: Log Transform
Try predicting log(Vs) instead of Vs directly. Compare RMSE on the original scale.

### Exercise 3: Feature Engineering
Add N1_60 (overburden-corrected N) as a feature. Does it improve predictions?

In [ ]:
# EXERCISE 2 STARTER:
# y_log = np.log(y)
# X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
#     X, y_log, test_size=0.2, random_state=42)
# rf_log = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
# rf_log.fit(scaler.fit_transform(X_train_l), y_train_l)
# y_pred_log = np.exp(rf_log.predict(scaler.transform(X_test_l)))
# y_test_orig = np.exp(y_test_l)
# print(f"Log-transform RMSE: {np.sqrt(mean_squared_error(y_test_orig, y_pred_log)):.1f} m/s")

---

## Key Takeaways

1. **Empirical correlations** (Imai & Tonouchi) use only N-value, limiting their accuracy for diverse soil conditions.

2. **ML models** leverage multiple features (depth, soil type, fines content) and consistently outperform single-variable empirical equations.

3. **Tree-based models** (RF, Gradient Boosting) typically perform best for Vs prediction due to their ability to capture nonlinear relationships.

4. **Residual analysis** reveals where predictions fail — often at extreme Vs values or specific soil types.

5. **Better Vs prediction directly improves seismic site classification**, which affects design ground motions and structural safety.

6. Even with ML, always **compare against empirical baselines** — if ML barely improves, the simpler method may be preferred.

---

## How This Connects to ML

| Today's Concept | ML Equivalent |
|----------------|---------------|
| Empirical equation (Vs = f(N)) | Univariate regression baseline |
| Multi-feature ML model | Multivariate regression |
| Model comparison table | Systematic evaluation framework |
| Residual analysis | Diagnostic plots for model debugging |
| Feature importance | Understanding model decisions |
| Site classification from Vs | Downstream task evaluation |

---

*Day 056 of 100 — ML hears what empirical equations miss!*